# 01. 데이터 개요 · 기술통계

가공 전 데이터(전국: `dataset.parquet`, 지역: `price.parquet`) 기준. **예측 대상은 소매가**라 소매 시계열만 본다. 중도매가는 피처(`whsl_price_kg` 등)로만 쓰이며, 축산물은 중도매 조사 자체가 없다(도매 쪽 정보는 축평원 경락가를 피처로).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parents[1]                     # demo/2_eda → 프로젝트 루트
P = ROOT / "data" / "processed"
NAMES = {"agri": "농산물", "livestock": "축산물", "fishery": "수산물"}
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200); pd.set_option("display.float_format", "{:,.3f}".format)

In [2]:
# 전국 소매가 + 모든 변수 (원본 값): data/processed/{분야}/dataset.parquet
# 예측 대상은 소매가 → 소매 시계열만. 중도매가는 피처(whsl_price_kg 등 열)로만 쓰임
ds = {d: pd.read_parquet(P / d / "dataset.parquet", filters=[("se_nm", "==", "소매")]) for d in NAMES}
ds = {d: df[df.date >= "2016-01-01"] for d, df in ds.items()}     # 2015년은 이력 계산용 준비 기간
{NAMES[d]: df.shape for d, df in ds.items()}

{'농산물': (284054, 116), '축산물': (62275, 111), '수산물': (66874, 136)}

## 1. 규모 — 품목·시계열·기간 (소매)

In [3]:
rows = []
for d, df in ds.items():
    rows.append({"분야": NAMES[d], "행 (전국·일)": len(df), "품목": df.item_nm.nunique(),
                 "소매 시계열 (품목·품종·등급)": df.groupby(["item_nm", "vrty_nm", "grd_nm"]).ngroups,
                 "기간": f"{df.date.min():%Y-%m-%d} ~ {df.date.max():%Y-%m-%d}"})
pd.DataFrame(rows)

,분야,행 (전국·일),품목,소매 시계열 (품목·품종·등급),기간
0,농산물,284054,70,210,2016-01-04 ~ 2026-09-29
1,축산물,62275,7,36,2016-01-04 ~ 2026-09-29
2,수산물,66874,27,63,2016-01-04 ~ 2026-09-29


## 2. 식재료 계열별 품목 수

In [4]:
pd.concat({NAMES[d]: df.groupby("food_group").item_nm.nunique() for d, df in ds.items()}, names=["분야", "계열"]).rename("품목 수").reset_index()

,분야,계열,품목 수
0,농산물,견과·종실류,4
1,농산물,곡류,5
2,농산물,과일,9
3,농산물,과일(가공),2
4,농산물,과일(수입),8
5,농산물,과채류,11
6,농산물,근채류,2
7,농산물,두류,3
8,농산물,버섯류,3
9,농산물,서류,2


## 3. 소매가 기술통계 (원/kg)

In [5]:
pd.DataFrame({NAMES[d]: df.price_kg.describe(percentiles=[.25, .5, .75]) for d, df in ds.items()}).T

,count,mean,std,min,25%,50%,75%,max
농산물,"284,054.000","8,147.293","7,197.147",357.143,"3,290.000","5,900.000","10,000.000","47,570.000"
축산물,"62,275.000","46,174.332","45,201.465",353.500,"5,746.000","39,995.000","69,585.000","197,460.000"
수산물,"66,874.000","21,149.696","19,556.778","1,332.000","8,002.047","15,000.000","27,722.222","119,866.667"


In [6]:
# 계열별 소매 중앙 가격 — 품목 간 가격 차이가 수백 배
pd.concat({NAMES[d]: df.groupby("food_group").price_kg.median().sort_values() for d, df in ds.items()}, names=["분야", "계열"]).rename("소매 중앙 가격 (원/kg)").reset_index()

,분야,계열,소매 중앙 가격 (원/kg)
0,농산물,근채류,"1,666.667"
1,농산물,곡류,"3,090.000"
2,농산물,엽채류,"4,050.000"
3,농산물,서류,"4,050.000"
4,농산물,과채류,"4,685.714"
5,농산물,과일(수입),"4,800.000"
6,농산물,버섯류,"5,000.000"
7,농산물,과일,"5,970.149"
8,농산물,양념채소,"9,900.000"
9,농산물,두류,"11,960.000"


## 4. 조사 빈도 — 시계열 하나가 1년에 며칠 조사되나

In [7]:
rows = []
for d, df in ds.items():
    n = df.groupby(["item_nm", "vrty_nm", "grd_nm", df.date.dt.year]).date.nunique()
    rows.append({"분야": NAMES[d], "연간 조사일 중앙값": n.median(), "하위 10%": n.quantile(.1), "상위 10%": n.quantile(.9)})
pd.DataFrame(rows)

,분야,연간 조사일 중앙값,하위 10%,상위 10%
0,농산물,224.000,28.000,246.000
1,축산물,280.000,16.000,366.000
2,수산물,242.000,57.100,246.000


## 5. 예측 대상 y

향후 28일 평균 소매가 / 최근 7일 평균 소매가 - 1 (`features.parquet`의 `y`, 전국)

In [8]:
y = {NAMES[d]: pd.read_parquet(P / d / "features.parquet", columns=["date", "y"], filters=[("se_nm", "==", "소매")]).query("date >= '2016-01-01'") for d in NAMES}
pd.DataFrame({k: v.y.describe(percentiles=[.01, .25, .5, .75, .99]) for k, v in y.items()}).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
농산물,"271,115.000",0.010,0.125,-0.646,-0.275,-0.038,0.000,0.040,0.459,1.858
축산물,"60,814.000",0.003,0.052,-0.468,-0.131,-0.019,0.000,0.021,0.165,0.716
수산물,"63,955.000",0.004,0.065,-0.411,-0.169,-0.016,0.000,0.020,0.219,1.006


## 6. 주요 외부 변수 기술통계 (전국, 날짜 단위)

In [9]:
cols = ["wx_temp_avg", "wx_rain", "wx_sunshine_hr", "wx_humidity", "usd_krw", "cpi_식료품", "fuel_보통휘발유", "intl_옥수수", "intl_대두"]
ds["agri"].drop_duplicates("date")[cols].describe().T

,count,mean,std,min,25%,50%,75%,max
wx_temp_avg,"2,634.000",13.739,9.616,-11.519,5.500,14.207,22.141,30.246
wx_rain,"2,634.000",3.632,8.916,0.000,0.005,0.212,2.185,103.246
wx_sunshine_hr,"2,634.000",6.374,3.257,0.000,4.047,6.921,8.784,12.920
wx_humidity,"2,634.000",69.321,12.798,32.652,59.342,70.514,79.217,95.584
usd_krw,"2,634.000","1,242.538",123.864,"1,057.600","1,134.100","1,196.950","1,340.925","1,554.400"
cpi_식료품,"2,634.000",106.970,13.508,85.892,94.497,105.160,120.910,128.990
fuel_보통휘발유,"2,629.000","1,585.217",164.916,"1,255.080","1,469.060","1,578.490","1,667.580","2,084.000"
intl_옥수수,"2,634.000",203.830,53.587,143.990,162.180,189.910,223.850,348.510
intl_대두,"2,634.000",412.029,85.448,305.310,339.960,380.870,471.930,621.200
